# 03 — Gold: fact_transaction

| Property | Value |
|----------|-------|
| **Gold Table** | `fact_transaction` |
| **Grain** | One row per TransactionId |
| **Source** | `rpt.vwTransaction` LEFT JOIN aggregated `rpt.vwTransactionDetailUSD` ON TransactionId |
| **PK** | `TransactionId` (int) |
| **Rows** | ~85,262,175 |

### What this notebook does:
1. Read `vwTransaction` (85M rows) — the header with all dimension FKs
2. Read `vwTransactionDetailUSD` (471M rows) — aggregate financials by TransactionId
3. LEFT JOIN header + aggregated detail
4. Add DateKey columns (YYYYMMDD integers)
5. DQ checks
6. Write to gold

> ⚠️ This is the biggest build. The aggregation collapses 471M → 85M rows.
> Carrier-level breakdowns are lost here but available in `fact_transaction_detail` (lakehouse only).

In [ ]:
# ============================================================
# Cell 1: Setup & Config
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")

# Optimize for large joins
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

LAKEHOUSE = "The_Global_Loom"
TABLE = "fact_transaction"
SOURCE_HEADER = "rpt.vwTransaction"
SOURCE_DETAIL = "rpt.vwTransactionDetailUSD"

print(f"✅ Config: {SOURCE_HEADER} + {SOURCE_DETAIL} → {LAKEHOUSE}.{TABLE}")

## Cell 2: Read transaction header (vwTransaction)

This is the header table with all dimension FKs (Policy, Product, Geography, Segment) but **no financial columns**.

In [ ]:
# ============================================================
# Cell 2: Read transaction header
# ============================================================
df_header = spark.table(SOURCE_HEADER)

header_count = df_header.count()
print(f"📥 Header: {header_count:,} rows × {len(df_header.columns)} cols")
df_header.printSchema()

## Cell 3: Read and aggregate transaction detail (vwTransactionDetailUSD)

Aggregate all 15 financial columns by TransactionId using SUM.

**Dropped financial columns** (all zero/empty): Adjustments, Discount, Expenses, WriteOff

**Kept**: GrossPremium, NetPremium, GrossBrokerage, NetBrokerage, GrossFee, NetFee, Claim, AdditionalCommission, ContingentCommission, MarketDerivedIncome, Deduction, CostOtherExpense, CostCompanyExpense, SharedBrokerage, SharedFee

In [ ]:
# ============================================================
# Cell 3: Read and aggregate transaction detail
# ============================================================
df_detail_raw = spark.table(SOURCE_DETAIL)

detail_count = df_detail_raw.count()
print(f"📥 Detail: {detail_count:,} rows × {len(df_detail_raw.columns)} cols")

# Aggregate financials by TransactionId
df_detail_agg = (
    df_detail_raw
    .groupBy("TransactionId")
    .agg(
        F.sum("GrossPremium").alias("GrossPremium"),
        F.sum("NetPremium").alias("NetPremium"),
        F.sum("GrossBrokerage").alias("GrossBrokerage"),
        F.sum("NetBrokerage").alias("NetBrokerage"),
        F.sum("GrossFee").alias("GrossFee"),
        F.sum("NetFee").alias("NetFee"),
        F.sum("Claim").alias("Claim"),
        F.sum("AdditionalCommission").alias("AdditionalCommission"),
        F.sum("ContingentCommission").alias("ContingentCommission"),
        F.sum("MarketDerivedIncome").alias("MarketDerivedIncome"),
        F.sum("Deduction").alias("Deduction"),
        F.sum("CostOtherExpense").alias("CostOtherExpense"),
        F.sum("CostCompanyExpense").alias("CostCompanyExpense"),
        F.sum("SharedBrokerage").alias("SharedBrokerage"),
        F.sum("SharedFee").alias("SharedFee"),
        F.count("TransactionDetailId").alias("DetailRowCount")
    )
)

agg_count = df_detail_agg.count()
print(f"\n✅ Aggregated: {detail_count:,} detail rows → {agg_count:,} transaction groups")
print(f"   Avg detail rows per transaction: {detail_count / agg_count:.1f}")

## Cell 4: Select header columns and join with aggregated detail

Pick the columns we need from the header, then LEFT JOIN the aggregated financials.

LEFT JOIN because ~4M headers have no detail rows.

In [ ]:
# ============================================================
# Cell 4: Select header columns + JOIN aggregated detail
# ============================================================

# Select header columns
df_header_clean = df_header.select(
    # --- Keys ---
    F.col("TransactionId").cast("int"),
    F.col("TransactionKey").cast("string"),
    F.col("DataSourceInstanceId").cast("int"),

    # --- Dimension FKs ---
    F.col("PolicyId").cast("int"),
    F.col("ProductId").cast("int"),
    F.col("GlobalFinancialGeographyId").cast("int"),
    F.col("GlobalFinancialSegmentId").cast("int"),
    F.col("GlobalLegalEntityId").cast("int"),

    # --- Denormalized party IDs ---
    F.col("ClientPartyId").cast("int"),
    F.col("InsuredPartyId").cast("int"),
    F.col("ReinsuredPartyId").cast("int"),

    # --- Denormalized product hierarchy ---
    F.col("GlobalProductClass").cast("string"),
    F.col("GlobalProductLine").cast("string"),
    F.col("GlobalProduct").cast("string"),

    # --- Dates ---
    F.col("TransactionDate").cast("timestamp"),
    F.col("InvoiceDate").cast("timestamp"),

    # --- DateKeys (YYYYMMDD integers) ---
    F.date_format(F.col("TransactionDate"), "yyyyMMdd").cast("int").alias("TransactionDateKey"),
    F.date_format(F.col("InvoiceDate"), "yyyyMMdd").cast("int").alias("InvoiceDateKey"),

    # --- Descriptive ---
    F.col("TransactionReference").cast("string"),
    F.col("SegmentCode").cast("string"),
    F.col("OwnershipOrganisation").cast("string"),

    # --- Flags ---
    F.col("IsDirectSettled").cast("boolean"),
    F.col("IsFee").cast("boolean"),
    F.col("IsDeleted").cast("boolean"),
    F.col("IsParentDeleted").cast("boolean")
)

print(f"✅ Header columns selected: {len(df_header_clean.columns)} cols")

# LEFT JOIN with aggregated detail
df_joined = (
    df_header_clean
    .join(
        df_detail_agg,
        on="TransactionId",
        how="left"
    )
)

join_count = df_joined.count()
print(f"✅ After JOIN: {join_count:,} rows × {len(df_joined.columns)} cols")
print(f"   Expected ~{header_count:,} (same as header — LEFT JOIN preserves all headers)")

## Cell 5: Final select with explicit types

Fill null financials with 0 (for headers with no detail rows) and set final column order.

In [ ]:
# ============================================================
# Cell 5: Final select
# ============================================================

# Financial columns to coalesce (null → 0)
financial_cols = [
    "GrossPremium", "NetPremium", "GrossBrokerage", "NetBrokerage",
    "GrossFee", "NetFee", "Claim", "AdditionalCommission",
    "ContingentCommission", "MarketDerivedIncome", "Deduction",
    "CostOtherExpense", "CostCompanyExpense", "SharedBrokerage", "SharedFee"
]

df_final = df_joined
for col_name in financial_cols:
    df_final = df_final.withColumn(
        col_name,
        F.coalesce(F.col(col_name), F.lit(0)).cast("decimal(18,4)")
    )

# Fill DetailRowCount null → 0
df_final = df_final.withColumn(
    "DetailRowCount",
    F.coalesce(F.col("DetailRowCount"), F.lit(0)).cast("int")
)

print(f"✅ Final: {df_final.count():,} rows × {len(df_final.columns)} cols")
df_final.printSchema()

## Cell 6: Data quality checks

In [ ]:
# ============================================================
# Cell 6: Data quality checks
# ============================================================
total = df_final.count()
dupes = total - df_final.select("TransactionId").distinct().count()
null_pk = df_final.filter(F.col("TransactionId").isNull()).count()
null_policy = df_final.filter(F.col("PolicyId").isNull()).count()
no_detail = df_final.filter(F.col("DetailRowCount") == 0).count()
has_detail = total - no_detail

# Financial totals for validation
from pyspark.sql import Row
financial_totals = df_final.agg(
    F.sum("GrossPremium").alias("TotalGrossPremium"),
    F.sum("NetPremium").alias("TotalNetPremium"),
    F.sum("GrossBrokerage").alias("TotalGrossBrokerage"),
    F.sum("Claim").alias("TotalClaim")
).collect()[0]

print(f"✅ DQ Checks")
print(f"   Total rows:            {total:,}")
print(f"   Duplicate PKs:         {dupes}")
print(f"   Null TransactionIds:   {null_pk}")
print(f"   Null PolicyIds:        {null_policy:,}")
print(f"   Headers with details:  {has_detail:,} ({has_detail*100/total:.1f}%)")
print(f"   Headers without:       {no_detail:,} ({no_detail*100/total:.1f}%)")
print(f"")
print(f"   Financial Totals (validation):")
print(f"     GrossPremium:   {financial_totals['TotalGrossPremium']:,.2f}")
print(f"     NetPremium:     {financial_totals['TotalNetPremium']:,.2f}")
print(f"     GrossBrokerage: {financial_totals['TotalGrossBrokerage']:,.2f}")
print(f"     Claim:          {financial_totals['TotalClaim']:,.2f}")

assert dupes == 0, f"❌ Found {dupes} duplicate TransactionIds!"
assert null_pk == 0, f"❌ Found {null_pk} null TransactionIds!"
print("\n✅ All DQ checks passed")

## Cell 7: Write to gold lakehouse

In [ ]:
# ============================================================
# Cell 7: Write to gold lakehouse
# ============================================================
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{LAKEHOUSE}.{TABLE}")

print(f"✅ Written: {LAKEHOUSE}.{TABLE}")
print(f"   Rows: {spark.table(f'{LAKEHOUSE}.{TABLE}').count():,}")